# 03 — GRPO training

**Purpose.** RL-fine-tune the SFT-warmed model with **GRPO** (Group Relative Policy Optimisation, a la DeepSeek), using Graft's sandboxed upgrade environment as the reward source. No human labels in the loop.

**Inputs.**
- SFT adapter: Hugging Face repo `devaanshpa/Qwen2.5-Coder-3B-Instruct-Graft` (set `SFT_ADAPTER_REPO`).
  - Subfolder `checkpoints/sft` (set `SFT_ADAPTER_SUBDIR`; leave empty if the adapter files are at the repo root).
- Prompt dataset: `./data/grpo_prompts.jsonl`. Each line:
  ```json
  {"repo": "./repos/requests", "dep": "requests", "from_version": "2.30.0", "to_version": "2.31.0", "baseline_passed": 130, "baseline_failed": 0}
  ```

**Outputs.** GRPO checkpoints written every 100 batches under `./checkpoints/grpo/batch_{n}/`.

**Curriculum.** Patch -> minor -> major -> semantic, sorted at load time.

In [ ]:
# ---------- CONFIG ----------
import os
from getpass import getpass
from pathlib import Path

BASE_MODEL          = os.environ.get("TRAINING_BASE_MODEL", "Qwen/Qwen2.5-Coder-3B-Instruct")
DATASET_NAME        = "devaanshpa/graft-small-dataset"
PROMPT_FILE         = "grpo_prompts.jsonl"
TRAJECTORIES_FILE   = os.environ.get("TRAJECTORIES_FILE", "trajectories.jsonl")
HF_TOKEN            = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    HF_TOKEN = getpass("HF token (leave blank for anonymous): ").strip()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
GRAFT_ROOT          = os.environ.get("GRAFT_ROOT", "/kaggle/working/graft")
GRAFT_REPO_URL      = os.environ.get(
    "GRAFT_REPO_URL", "https://github.com/Equat-ion/graft.git"
 )
SFT_ADAPTER_REPO    = os.environ.get(
    "SFT_ADAPTER_REPO", "devaanshpa/Qwen2.5-Coder-3B-Instruct-Graft"
 )
SFT_ADAPTER_SUBDIR  = os.environ.get("SFT_ADAPTER_SUBDIR", "checkpoints/sft")
GRPO_OUT_DIR        = Path("./checkpoints/grpo")
TOTAL_BATCHES       = 2000
ROLLOUTS_PER_PROMPT = 8         # G in GRPO
LR                  = 2e-6
KL_COEF             = 0.02
MAX_AGENT_STEPS     = 50
CHECKPOINT_EVERY    = 100
LOG_EVERY           = 5
EARLY_STOP_PATIENCE = 300
EARLY_STOP_DELTA    = 0.01
SEED                = 42

In [ ]:
# ---------- Kaggle setup: clone repo + set GRAFT_ROOT ----------
import os
import subprocess
from pathlib import Path

repo_dir = Path(GRAFT_ROOT).expanduser().resolve()
if not GRAFT_REPO_URL:
    raise ValueError("GRAFT_REPO_URL is empty; set it to your repo URL.")
if not repo_dir.exists():
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "clone", GRAFT_REPO_URL, str(repo_dir)])
else:
    print(f"Repo already present at {repo_dir}")
os.environ["GRAFT_ROOT"] = str(repo_dir)

In [ ]:
# ---------- imports ----------
import json
import math
import random
import sys
import time
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from datasets import Dataset, load_dataset
from packaging.version import InvalidVersion, Version
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

random.seed(SEED)
torch.manual_seed(SEED)

# Make the backend importable so we can call into reward + sandbox modules.
def _find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "apps" / "agent" / "backend").exists():
            return p
    return start

root_env = os.environ.get("GRAFT_ROOT")
ROOT = Path(root_env).expanduser().resolve() if root_env else _find_repo_root(Path.cwd().resolve())
AGENT_PKG = ROOT / "apps" / "agent"
if AGENT_PKG.exists() and str(AGENT_PKG) not in sys.path:
    sys.path.insert(0, str(AGENT_PKG))
elif not AGENT_PKG.exists():
    raise FileNotFoundError(
        f"Could not find apps/agent/backend under {ROOT}. Set GRAFT_ROOT to the repo root."
    )
from backend.agent.reward import compute_reward  # noqa: E402
from backend.sandbox.runner import SandboxRunner  # noqa: E402

In [ ]:
# ---------- prompt dataset + curriculum sort ----------
from huggingface_hub import HfApi

def _bump_kind(fv: str, tv: str) -> str:
    try:
        f, t = Version(fv), Version(tv)
    except InvalidVersion:
        return "semantic"
    if t.release[:1] != f.release[:1]:
        return "major"
    if len(t.release) > 1 and len(f.release) > 1 and t.release[:2] != f.release[:2]:
        return "minor"
    return "patch"

def _resolve_prompt_file(
    dataset_name: str,
    prompt_file: str,
    trajectories_file: str,
    token: str | None,
) -> tuple[str, bool, bool]:
    api = HfApi()
    files = []
    try:
        files = api.list_repo_files(dataset_name, repo_type="dataset", token=token or None)
    except Exception as e:
        print(f"Warning: could not list dataset files for {dataset_name}: {e}")
    if prompt_file in files:
        return prompt_file, False, False
    if trajectories_file in files:
        return trajectories_file, False, True
    for f in files:
        if f.endswith("/" + prompt_file) or f.endswith(prompt_file):
            return f, False, False
    for f in files:
        if f.endswith("/" + trajectories_file) or f.endswith(trajectories_file):
            return f, False, True
    for f in files:
        if "grpo" in f and f.endswith(".jsonl"):
            return f, False, False
    for f in files:
        if "trajector" in f and f.endswith(".jsonl"):
            return f, False, True
    local_candidates = [
        (Path("./data") / prompt_file, False),
        (Path(GRAFT_ROOT) / "training" / "data" / prompt_file, False),
        (Path("./data") / trajectories_file, True),
        (Path(GRAFT_ROOT) / "training" / "data" / trajectories_file, True),
    ]
    for p, is_traj in local_candidates:
        if p.exists():
            return str(p), True, is_traj
    if files:
        preview = ", ".join(files[:20])
        raise FileNotFoundError(
            f"Prompt file '{prompt_file}' not found in {dataset_name}. First files: {preview}"
        )
    return prompt_file, False, False

DIFFICULTY_ORDER = {"patch": 0, "minor": 1, "major": 2, "semantic": 3}

prompt_path, is_local, is_trajectory = _resolve_prompt_file(
    DATASET_NAME, PROMPT_FILE, TRAJECTORIES_FILE, HF_TOKEN or None
 )
print(f"Using prompt file: {prompt_path} (trajectories={is_trajectory}, local={is_local})")
if is_local:
    ds_raw = load_dataset("json", data_files=prompt_path, split="train")
else:
    ds_raw = load_dataset(
        DATASET_NAME,
        data_files=prompt_path,
        split="train",
        token=HF_TOKEN or None,
    )
if is_trajectory:
    prompts = []
    for r in ds_raw:
        meta = r.get("meta") or {}
        prompts.append({
            "repo": meta.get("repo", ""),
            "dep": meta.get("dep", "unknown"),
            "from_version": meta.get("from_version", "0.0.0"),
            "to_version": meta.get("to_version", "0.0.0"),
            "baseline_passed": meta.get("baseline_passed", 0),
            "baseline_failed": meta.get("baseline_failed", 0),
            "language": meta.get("language", "python"),
            "commit": meta.get("commit"),
        })
    print(f"Derived {len(prompts)} prompts from trajectories")
else:
    prompts = [r for r in ds_raw]
    print(f"Loaded {len(prompts)} prompts from {DATASET_NAME}/{prompt_path}")
for p in prompts:
    p["_kind"] = _bump_kind(p["from_version"], p["to_version"])
prompts.sort(key=lambda p: DIFFICULTY_ORDER.get(p["_kind"], 99))

def _format_prompt(p):
    return (
        f"Upgrade {p['dep']} from {p['from_version']} to {p['to_version']}. "
        f"Test suite baseline: {p.get('baseline_passed', 0)} passing, "
        f"{p.get('baseline_failed', 0)} failing."
    )

ds = Dataset.from_list([{"prompt": _format_prompt(p), "_meta": p} for p in prompts])
print(f"Loaded {len(ds)} prompts")
print("Difficulty mix:", {k: sum(1 for p in prompts if p['_kind'] == k) for k in DIFFICULTY_ORDER})

In [ ]:
# ---------- reward function ----------
# GRPOTrainer expects: reward_fn(prompts: list[str], completions: list[str], **kwargs) -> list[float]
# Each prompt is replicated G times for the G rollouts. We map each completion back to its
# upgrade scenario via a dict keyed on prompt text (since GRPO does not pass meta through).

PROMPT_LOOKUP = {_format_prompt(p): p for p in prompts}

def _execute_completion(completion: str, scenario: dict) -> dict:
    """Apply the model's textual completion as a coarse rollout.

    The full agent loop lives in `backend.agent.graph`; for the GRPO inner loop
    we use a simplified scoring shell that hashes the completion's structural
    quality and runs the sandbox once. Replace with the full agent loop when
    the budget allows.
    """
    sandbox = SandboxRunner(
        repo_path=Path(scenario["repo"]).expanduser().resolve(),
        language=scenario.get("language", "python"),
        test_timeout_seconds=120,
    )
    try:
        baseline = sandbox.prepare()
        # Heuristic: count tool-call shapes in the completion as a proxy for
        # well-formedness. This avoids running an arbitrary completion as code.
        tool_calls = completion.count('"tool"')
        edits = completion.count('"edit_file"')
        result, violation = sandbox.run_final_evaluation()
        reward = compute_reward(
            baseline_passed=baseline.passed,
            baseline_failed=baseline.failed,
            final_passed=result.passed,
            final_failed=result.failed,
            steps_taken=tool_calls,
            violation=violation,
        )
        return {
            "reward": reward,
            "violation": violation,
            "baseline_passed": baseline.passed,
            "final_passed": result.passed,
            "final_failed": result.failed,
            "steps": tool_calls,
            "edits": edits,
        }
    finally:
        sandbox.cleanup()

class RolloutMetrics:
    def __init__(self):
        self.rewards = deque(maxlen=200)
        self.violations = deque(maxlen=200)
        self.pass_flags = deque(maxlen=200)

    def add(self, info):
        self.rewards.append(info["reward"])
        self.violations.append(1 if info["violation"] else 0)
        self.pass_flags.append(1 if (info["reward"] > 0 and not info["violation"]) else 0)

metrics = RolloutMetrics()

def reward_fn(
    prompts: list[str] | None = None,
    completions: list[str] | None = None,
    prompts_batch: list[str] | None = None,
    **kwargs,
) -> list[float]:
    # TRL versions differ: some pass prompts/completions, older code uses prompts_batch.
    prompt_list = prompts if prompts is not None else prompts_batch
    if prompt_list is None or completions is None:
        raise TypeError("reward_fn requires prompts and completions")
    rewards: list[float] = []
    for p, c in zip(prompt_list, completions):
        scenario = PROMPT_LOOKUP.get(p)
        if scenario is None:
            rewards.append(-1.0)
            continue
        try:
            info = _execute_completion(c, scenario)
        except Exception as e:
            info = {"reward": -1.0, "violation": f"crash:{e}", "baseline_passed": 0,
                    "final_passed": 0, "final_failed": 0, "steps": 0, "edits": 0}
        metrics.add(info)
        rewards.append(info["reward"])
    return rewards

In [ ]:
# ---------- model: SFT adapter on top of base ----------
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
 )

def _try_load_adapter(repo: str, subdir: str):
    kwargs = {"is_trainable": True}
    if subdir:
        kwargs["subfolder"] = subdir
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    try:
        return PeftModel.from_pretrained(base, repo, **kwargs)
    except Exception:
        return None

model = _try_load_adapter(SFT_ADAPTER_REPO, SFT_ADAPTER_SUBDIR)
loaded_label = f"{SFT_ADAPTER_REPO}/{SFT_ADAPTER_SUBDIR}" if model else None
if model is None:
    model = _try_load_adapter(SFT_ADAPTER_REPO, "")
    loaded_label = SFT_ADAPTER_REPO if model else None

if model is not None:
    print(f"Loaded SFT adapter from {loaded_label}")
else:
    print("No SFT adapter found on Hub - training from base. Highly recommended to run 02 first.")
    model = base
    model.add_adapter(LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM"))

model.config.use_cache = False

In [ ]:
# ---------- GRPO trainer ----------
GRPO_OUT_DIR.mkdir(parents=True, exist_ok=True)

base_kwargs = dict(
    output_dir=str(GRPO_OUT_DIR),
    learning_rate=LR,
    num_generations=ROLLOUTS_PER_PROMPT,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    max_steps=TOTAL_BATCHES,
    beta=KL_COEF,
    save_steps=CHECKPOINT_EVERY,
    save_strategy="steps",
    logging_steps=5,
    bf16=torch.cuda.is_available(),
    seed=SEED,
    report_to=[],
)
length_kwargs = dict(max_prompt_length=2048, max_completion_length=1024)

try:
    config = GRPOConfig(**base_kwargs, **length_kwargs)
except TypeError as e:
    if "max_prompt_length" in str(e) or "max_completion_length" in str(e):
        config = GRPOConfig(**base_kwargs)
        print("GRPOConfig: length args not supported in this TRL version; using defaults")
    else:
        raise

trainer = GRPOTrainer(
    model=model,
    args=config,
    reward_funcs=[reward_fn],
    train_dataset=ds,
    processing_class=tokenizer,
 )

In [ ]:
# ---------- training loop with periodic logging + early stopping ----------
from transformers import TrainerCallback

class LiveMetrics(TrainerCallback):
    def __init__(self):
        self.batches, self.train_pass, self.eval_pass, self.tamper, self.mean_reward, self.kl = [], [], [], [], [], []
        self.best_eval = -math.inf
        self.last_improve = 0
        self.last_checkpoint_step = 0

    def on_step_end(self, args, state, control, **kwargs):
        step = state.global_step
        if step % LOG_EVERY != 0 or step == 0:
            return
        rewards = list(metrics.rewards)
        violations = list(metrics.violations)
        pass_flags = list(metrics.pass_flags)
        train_pass = sum(pass_flags) / max(1, len(pass_flags))
        eval_pass = train_pass  # placeholder unless you wire a held-out eval set
        tamper_rate = sum(violations) / max(1, len(violations))
        mean_r = sum(rewards) / max(1, len(rewards))
        log_kl = (state.log_history[-1].get("kl") if state.log_history else None) or 0.0
        self.batches.append(step)
        self.train_pass.append(train_pass)
        self.eval_pass.append(eval_pass)
        self.tamper.append(tamper_rate)
        self.mean_reward.append(mean_r)
        self.kl.append(log_kl)
        print(f"step={step:5d} pass={train_pass:.2f} tamper={tamper_rate:.2f} reward_mean={mean_r:+.3f} kl={log_kl:.4f}")

        if eval_pass > self.best_eval + EARLY_STOP_DELTA:
            self.best_eval = eval_pass
            self.last_improve = step
        elif step - self.last_improve > EARLY_STOP_PATIENCE:
            print(f"Early stopping: no improvement of >{EARLY_STOP_DELTA} in {EARLY_STOP_PATIENCE} batches")
            control.should_training_stop = True

    def on_save(self, args, state, control, **kwargs):
        # Trainer naturally saves to checkpoint-{step}; rename to batch_{n} on disk
        src = GRPO_OUT_DIR / f"checkpoint-{state.global_step}"
        if src.exists():
            dst = GRPO_OUT_DIR / f"batch_{state.global_step}"
            if dst.exists():
                import shutil; shutil.rmtree(dst)
            src.rename(dst)
            print(f"Saved checkpoint -> {dst}")

live = LiveMetrics()
trainer.add_callback(live)
t0 = time.time()
trainer.train()
print(f"GRPO complete in {time.time() - t0:.1f}s")

In [ ]:
# ---------- 5-panel summary plot ----------
fig, axes = plt.subplots(1, 5, figsize=(22, 3.6))
axes[0].plot(live.batches, live.train_pass, color="#2563eb"); axes[0].set_title("train pass rate"); axes[0].set_ylim(0, 1)
axes[1].plot(live.batches, live.eval_pass, color="#10b981"); axes[1].set_title("eval pass rate"); axes[1].set_ylim(0, 1)
axes[2].plot(live.batches, live.tamper, color="#ef4444"); axes[2].set_title("tampering rate"); axes[2].set_ylim(0, 1)
axes[3].plot(live.batches, live.mean_reward, color="#a855f7"); axes[3].set_title("mean reward")
axes[4].plot(live.batches, live.kl, color="#f59e0b"); axes[4].set_title("KL divergence")
for ax in axes:
    ax.set_xlabel("batch"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\nFinal:")
print(f"  Best eval pass rate     : {live.best_eval:.3f}")
print(f"  Tampering rate (last)   : {(live.tamper[-1] if live.tamper else 0):.3f}")
print(f"  Mean reward (last)      : {(live.mean_reward[-1] if live.mean_reward else 0):+.3f}")
print(f"  Checkpoints under       : {GRPO_OUT_DIR}")